In [9]:

"""
SFLASH (C*-Minus) İmzalama Şeması - Referans Uygulama
========================================================

Bu modül, Mustafa SOLMAZ'ın "Kuantum Sonrası Çok Değişkenli Açık Anahtar
Kriptosistemlerinin İncelenmesi" başlıklı yüksek lisans tezinin 5.1.5.1
bölümünde ("SFLASH / C*-Minus Kriptosistemi") teorik olarak sunulan
imzalama şemasının SageMath ortamında somutlaştırılmış halidir.

Teorik Arka Plan
-----------------
SFLASH, Matsumoto-Imai (MI / C*) merkez dönüşümünü temel cisim, ancak bir
imzalama şeması olarak kullanılabilmesi için "Minus" (eksiltme)
dönüşümüyle güçlendirilmiş bir Big-Field ailesi kriptosistemidir.

Standart MI şemasının şifreleme amaçlı kullanımı, Patarin'in
lineerleştirme (linearization equations) saldırısı ile kırılmıştır. Bu
saldırının özü, MI merkez dönüşümü F(Y) = Y^(1+q^theta)'nın E genişleme
cismi üzerinde çok katı bir cebirsel yapıya sahip olması ve bu yapının
açık anahtarda x (girdi) ile y = P(x) (çıktı) arasında saldırgan
tarafından kolayca kurulabilen bilineer (lineer) denklemler üretmesidir.

SFLASH'ın MI'dan farkı, "Minus" (-) dönüşümüdür: açık anahtarın ürettiği
n adet polinomdan r tanesi kasıtlı olarak YAYINLANMAZ (gizlenir). Bu
eksiltme, saldırganın Patarin'in lineerleştirme denklemlerini eksiksiz
kuramamasına yol açar; çünkü denklem sisteminin bir kısmı saldırgan
tarafından bilinmemektedir. Böylece, imzalama bağlamında (şifreleme
değil) SFLASH, MI'nın klasik lineerleştirme saldırısına karşı
belirli ölçüde dayanıklı hale getirilmiş olur. (SFLASH, NESSIE projesi
tarafından bir süre önerilmiş, ancak daha sonra farklı bir cebirsel
saldırıyla -- diferansiyel simetri saldırısı -- kırılmıştır; bu tarihsel
bağlam tezin ilgili bölümünde ayrıntılı olarak ele alınmıştır.)

Sistemin matematiksel bileşenleri:
    - F_q          : q elemanlı sonlu cisim
    - F_q[x1..xn]   : Açık anahtarın (eksiltmeden önce) tanımlı olduğu
                       n değişkenli polinom halkası
    - E = F_q^n     : MI çekirdek dönüşümünün çalıştığı genişleme cismi
                       (F_q üzerinde n. dereceden bir indirgenemez
                       polinomla inşa edilir)
    - S, T          : Gizli, tersinir afin dönüşümler
    - F(Y) = Y^e    : MI merkez dönüşümü (e = 1 + q^theta)
    - "Minus"       : Açık anahtarın son r bileşeninin atılması
                       (Public_Key_Minus = Public_Key_Full[: n - r])

İmzalama şemasının genel akışı (bipolar kurulum, tez Bölüm 3.2.1.1'e
paralel olarak) şu şekildedir:
    1. İmzalanacak mesaj (n - r boyutlu) alınır.
    2. Eksik r bileşen rastgele "padding" (dolgu) değerleriyle
       tamamlanarak n boyutlu bir hedef vektör (y_full) elde edilir.
    3. Sırasıyla S^-1, MI^-1 (=Y^d), T^-1 dönüşümleri uygulanarak
       imza (x) hesaplanır.
    4. Doğrulama, imza açık anahtara (Minus uygulanmış haliyle)
       sokularak orijinal mesajın elde edilip edilmediğinin kontrol
       edilmesiyle yapılır.

Bu betik, küçük parametrelerle (GF(4), n=5, r=2) çalışan somut bir
örnek üzerinden anahtar üretimi, imzalama ve doğrulama adımlarını
ayrıntılı  biçimde izlenebilir kılmak amacıyla hazırlanmıştır.

Referans: Tez, Bölüm 5 — BigField Tabanlı CDPT Kriptosistemleri
"""

from sage.all import *


class SFLASH:
    """
    SFLASH (C*-Minus) imzalama şemasının ayrıntılı SageMath
    uygulaması.

    Bu sınıf, tezin 5.1.5.1 bölümünde anlatılan SFLASH kurulumunu adım
    adım gerçekleştirir: gizli anahtar bileşenlerinin (S, T) üretilmesi,
    MI merkez dönüşümünün genişleme cismi üzerinde sembolik olarak
    hesaplanması, açık anahtarın "Minus" dönüşümüyle inşası, imzalama
    ve doğrulama.

    Matematiksel Kurulum
    ---------------------
    1. Temel cisim F = GF(q) ve n değişkenli polinom halkası
       R = F[x1,...,xn].
    2. MI çekirdeğinin çalıştığı genişleme cismi E = F_{q^n}; F üzerinde
       n. dereceden indirgenemez bir polinomla (irr_poly) inşa edilir.
       F_q^n vektör uzayı ile E cismi arasında bir vektör uzayı
       izomorfizması kurulur (bkz. extension_to_vector /
       vector_to_extension metodları).
    3. MI üsleri: e = 1 + q^theta olmak üzere, F(Y) = Y^e dönüşümünün
       E* üzerinde bijektif (bire-bir) olabilmesi için
       gcd(e, q^n - 1) = 1 koşulunun sağlanması gerekir. Bu koşul
       sağlandığında d = e^{-1} mod (q^n - 1) hesaplanarak imzalamada
       kullanılacak tersinir üs elde edilir.
    4. "Minus" dönüşümü: n bileşenli tam açık anahtardan son r bileşen
       atılarak (n - r) bileşenli, fiilen yayınlanan açık anahtar elde
       edilir. Bu eksiltme, saldırganın sistemin tüm cebirsel yapısına
       erişimini kısıtlayarak lineerleştirme tipi saldırılara karşı
       ek bir savunma katmanı oluşturur.

    Parametreler
    ------------
    q : int
        Sonlu cismin eleman sayısı.
    n : int
        Değişken sayısı / genişleme cisminin F_q üzerindeki derecesi.
        gcd(1 + q^theta, q^n - 1) = 1 koşulunu sağlayacak şekilde
        seçilmelidir.
    theta : int
        MI merkez dönüşümündeki üs parametresi (e = 1 + q^theta).
    r : int
        "Minus" işleminde silinecek (yayınlanmayacak) denklem sayısı.
        İmzalama sırasında bu r kadar bileşen, rastgele padding
        (dolgu) değerleriyle tamamlanır.

    Öznitelikler (Anahtar Bileşenleri)
    ------------------------------------
    S_mat, S_vec : Gizli afin dönüşüm S(w) = S_mat * w + S_vec
    T_mat, T_vec : Gizli afin dönüşüm T(x) = T_mat * x + T_vec
    Public_Key_Minus : "Minus" uygulanmış, fiilen yayınlanan açık
                        anahtar; (n - r) bileşenli kuadratik polinom
                        sistemi.

    Notlar
    ------
    Bu sınıf, öğretim ve doğrulama amaçlı ayrıntılı  çıktı
    üretir; performans odaklı bir üretim ortamı implementasyonu değildir.
    """

    def __init__(self, q, n, theta, r):
        """
        SFLASH (C*-) Simülasyonu - Aritmetik Detaylı
        q: Cisim boyutu
        n: Değişken sayısı
        theta: MI üssü parametresi
        r: Minus (Silinen denklem) sayısı

        Bu kurucu metot, sistemin çalışacağı tüm cebirsel yapıları
        (temel cisim, polinom halkası, genişleme cismi) kurar ve MI
        üslerinin (e, d) tutarlılığını doğrular. Gizli anahtar
        bileşenleri (S, T) burada henüz üretilmez; bunlar
        generate_keys() metodunda oluşturulur.

        Parametreler
        ------------
        q : int
            Sonlu cisim boyutu.
        n : int
            Değişken sayısı / genişleme cisminin derecesi.
        theta : int
            MI merkez dönüşümü üs parametresi.
        r : int
            Minus işleminde silinecek denklem sayısı.

        Fırlatılan Hatalar
        -------------------
        ValueError
            gcd(1 + q^theta, q^n - 1) != 1 olduğunda, yani MI merkez
            dönüşümünün E* üzerinde tersinir (bijektif) olmadığı
            durumda fırlatılır.
        """
        self.q = q
        self.n = n
        self.theta = theta
        self.r = r
        
        # 1. Temel Cisim ve Halka
        # F: q elemanlı sonlu cisim. R: açık anahtarın (Minus'tan önce)
        # tanımlı olacağı n değişkenli polinom halkası F[x1,...,xn].
        self.F = GF(q, 'a')
        self.R = PolynomialRing(self.F, n, 'x')
        self.vars = vector(self.R, self.R.gens())
        
        # 2. Genişleme Cismi (MI Çekirdeği)
        # E = F[X] / (irr_poly)
        # MI merkez dönüşümü F(Y) = Y^e, F_q üzerinde değil, F_{q^n}
        # genişleme cisminde tanımlanır. Bu nedenle önce F_q üzerinde
        # n. dereceden indirgenemez bir polinom (irr_poly) seçilir,
        # ardından E = F_q[X] / (irr_poly) bölüm halkası (bir cisim)
        # inşa edilir.
        self.R_uni = PolynomialRing(self.F, 'X')
        self.irr_poly = self.R_uni.irreducible_element(n)
        self.E = self.F.extension(self.irr_poly, 'w') # 'w' genişleme cisminin üretecidir
        
        print("\n" + "="*80)
        print(f"SFLASH SİSTEMİ: ")
        print(f"Parametreler: GF({q}), n={n}, theta={theta}, r={r}")
        print(f"Genişleme Cismi: F_{q}^{n} (Üreteç 'w', Polinom: {self.irr_poly})")
        print("="*80)
        
        # MI Üsleri
        # e = 1 + q^theta: MI merkez dönüşümünün üssü. Bu dönüşümün
        # E* (E'nin sıfırdan farklı elemanlarının çarpımsal grubu)
        # üzerinde bir bijeksiyon tanımlayabilmesi için
        # gcd(e, q^n - 1) = 1 olması ZORUNLUDUR; aksi halde tersi (d)
        # hesaplanamaz ve imza üretimi (dolayısıyla MI'nin tersi)
        # mümkün olmaz.
        self.e = 1 + q**theta
        try:
            # d, e'nin (q^n - 1) modülüne göre çarpımsal tersidir.
            # d, imzalama aşamasında R = W^d işlemiyle MI dönüşümünün
            # tersini almak için kullanılır.
            self.d = inverse_mod(self.e, q**n - 1)
            print(f"[KURULUM] Gizli Üs e={self.e}")
            print(f"[KURULUM] Ters Üs  d={self.d} (GCD=1)")
        except:
            raise ValueError(f"HATA: GCD(1+q^theta, q^n-1) != 1.")

        # Gizli ve türetilmiş anahtar bileşenleri; generate_keys()
        # çağrılana kadar tanımsız (None) bırakılır.
        self.S_mat = None; self.S_vec = None
        self.T_mat = None; self.T_vec = None
        self.Public_Key_Minus = None

    # --- YARDIMCI FONKSİYONLAR ---
    def extension_to_vector(self, element):
        """
        Genişleme cismi E'ye ait bir elemanı, F_q^n vektör uzayındaki
        karşılığına dönüştürür.

        Teorik Gerekçe
        ---------------
        E = F_q[X] / (irr_poly) bölüm halkası, F_q üzerinde n boyutlu
        bir vektör uzayı yapısına da sahiptir (taban: 1, w, w^2, ...,
        w^(n-1)). Bu metot, E cismindeki bir elemanı önce F_q
        üzerindeki polinom temsiline (lift/polynomial) indirger,
        ardından bu polinomun katsayılarını n boyutlu bir vektöre
        (F_q^n) yerleştirir. Bu dönüşüm, MI çekirdeğinin ürettiği
        sonuçların (veya imzalamada ters MI işleminin sonucunun)
        F_q^n uzayına geri taşınması için gereklidir.

        Parametreler
        ------------
        element : E cismi elemanı (veya E'ye gömülü bir ifade)
            Vektöre dönüştürülecek genişleme cismi elemanı.

        Dönüş Değeri
        ------------
        vector
            F_q üzerinde tanımlı, uzunluğu n olan katsayı vektörü.
        """
        # E elemanını (w cinsinden polinom) -> F vektörüne çevirir
        try: poly = element.lift()
        except: 
            try: poly = element.polynomial()
            except: poly = element
        poly = self.R_uni(poly)
        coeffs = poly.list()
        # Katsayı listesi n'den kısa ise (yüksek dereceli terimler
        # sıfır olduğu için), eksik kısımlar 0 ile doldurulur.
        while len(coeffs) < self.n: coeffs.append(self.F(0))
        return vector(self.F, coeffs)

    def vector_to_extension(self, vec):
        """
        F_q^n uzayındaki bir vektörü, genişleme cismi E'ye ait
        karşılığına dönüştürür (extension_to_vector'ın tersi).

        Teorik Gerekçe
        ---------------
        vec = (v0, v1, ..., v_{n-1}) vektörü, E cisminin w tabanına
        göre sum(v_i * w^i) şeklinde yorumlanarak tek bir E elemanına
        eşlenir. Bu işlem, MI merkez dönüşümünün (veya tersinin) Y^e
        (Y^d) biçiminde genişleme cismi üzerinde uygulanabilmesi için,
        F_q^n vektörlerinin önce E'ye taşınmasını sağlar.

        Parametreler
        ------------
        vec : vector
            F_q üzerinde tanımlı, uzunluğu n olan vektör.

        Dönüş Değeri
        ------------
        E cismi elemanı
            vec vektörünün E cismindeki karşılığı.
        """
        # F vektörünü -> E elemanına (w cinsinden polinom) çevirir
        w_gen = self.E.gen()
        res = self.E(0)
        for i in range(self.n):
            res += vec[i] * (w_gen**i)
        return res
    
    def reduce_F_ideal(self, polys):
        """
        Verilen polinom vektörünü, F_q "cisim idealine" (field ideal)
        göre indirger.

        Teorik Gerekçe
        ---------------
        F_q sonlu cisminde her a elemanı için a^q = a özdeşliği
        geçerlidir. Bu nedenle x_i^q - x_i biçimindeki polinomlar,
        F_q^n üzerindeki tüm noktalarda sıfır olur ve bu polinomların
        ürettiği ideal (cisim ideali) ile indirgeme yapmak, polinomların
        F_q^n üzerinde aynı fonksiyonu temsil eden ama daha düşük
        dereceli/normalize edilmiş bir temsilini elde etmeyi sağlar.
        Açık anahtarın kuadratik (derece <= 2) bir sistem olarak
        kalmasını garanti altına almak için bu indirgeme kritik önem
        taşır.

        Parametreler
        ------------
        polys : iterable
            İndirgenecek polinomlar listesi/vektörü.

        Dönüş Değeri
        ------------
        vector
            cisim idealine göre indirgenmiş polinomlardan oluşan vektör.
        """
        field_eqs = [self.vars[i]**self.q - self.vars[i] for i in range(self.n)]
        FieldIdeal = self.R.ideal(field_eqs)
        return vector(self.R, [FieldIdeal.reduce(p) for p in polys])

    def generate_keys(self):
        """
        SFLASH gizli anahtarlarını (S, T) üretir, MI merkez
        dönüşümünü genişleme cismi üzerinde sembolik olarak
        hesaplayarak tam açık anahtarı elde eder ve "Minus" işlemiyle
        fiilen yayınlanacak açık anahtarı (Public_Key_Minus) oluşturur.

        Algoritmanın Genel Akışı
        --------------------------
        Tezde (5.1.5.1) tanımlanan kurulum şu adımları izler:
            1. Gizli, tersinir afin dönüşümler S ve T seçilir.
            2. MI merkez dönüşümü F(Y) = Y^e, x değişkenleri cinsinden
               sembolik olarak genişleme cismi E üzerinde hesaplanır:
                   y = T(x)  (F_q^n üzerinde afin dönüşüm)
                   Y(x) = y'nin E cismindeki sembolik temsili
                   Z(x) = Y(x)^e
               Z(x), F_q'nun cisim özdeşliği (x_i^q = x_i) kullanılarak
               indirgenir ve E katsayıları tek tek F_q^n'e geri
               açılarak MI_vec (n bileşenli kuadratik polinom vektörü)
               elde edilir.
            3. Tam açık anahtar:
                   Public_Key_Full = S(MI_vec) = S_mat * MI_vec + S_vec
               hesaplanır ve cisim idealine göre indirgenir.
            4. "MINUS" işlemi: Public_Key_Full'un son r bileşeni atılır;
               geriye kalan (n - r) bileşen fiilen yayınlanan açık
               anahtardır (Public_Key_Minus). Bu eksiltme, MI'nin
               klasik lineerleştirme saldırısına karşı ek bir koruma
               katmanı sağlar, çünkü saldırgan artık sistemin
               tamamına (n denklemin tümüne) erişemez.

        Dönüş Değeri
        ------------
        vector
            "Minus" uygulanmış, fiilen yayınlanan açık anahtar;
            (n - r) bileşenli kuadratik polinom sistemi.
        """
        print("\n" + "*"*40 + " ADIM 1: ANAHTAR ÜRETİMİ  " + "*"*40)
        
        # 1. AFİN DÖNÜŞÜMLER
        # T ve S, sırasıyla girdi (x) ve MI çekirdeğinin çıktısı
        # üzerinde uygulanan gizli, TERSİNİR afin dönüşümlerdir.
        # Tersinirlik şartı, imzalama sırasında S^-1 ve T^-1
        # dönüşümlerinin var olabilmesi için zorunludur.
        while True:
            self.T_mat = random_matrix(self.F, self.n, self.n)
            if self.T_mat.is_invertible(): break
        self.T_vec = random_vector(self.F, self.n)
        
        while True:
            self.S_mat = random_matrix(self.F, self.n, self.n)
            if self.S_mat.is_invertible(): break
        self.S_vec = random_vector(self.F, self.n)
        
        print(f"\n[1.1] Gizli Afin Dönüşümler:")
        print(f"   T Matrisi ({self.n}x{self.n}):\n{self.T_mat}")
        print(f"   S Matrisi ({self.n}x{self.n}):\n{self.S_mat}")
        print(f"   c_T (Afin Sabit Vektörü): {self.T_vec}")
        print(f"   c_S (Afin Sabit Vektörü): {self.S_vec}")

        # 2. AÇIK ANAHTAR İNŞASI
        print(f"\n[1.2] Polinom Aritmetiği: P = S o MI o T")
        
        # A) T(x) Hesabı
        # y = T(x): x değişkenleri üzerinde sembolik olarak hesaplanan
        # afin dönüşüm sonucu.
        y_vec = self.T_mat * self.vars + self.T_vec
        
        # B) Genişleme Cismine Geçiş (Lifting)
        # x değişkenlerini içeren bir E elemanı oluşturuyoruz
        # MI çekirdeğinin sembolik olarak (x değişkenleri cinsinden)
        # hesaplanabilmesi için, y_vec bileşenleri genişleme cismi E
        # üzerindeki çok değişkenli bir polinom halkasına taşınır.
        R_E_multi = PolynomialRing(self.E, self.n, 'x')
        y_vec_E = vector(R_E_multi, [R_E_multi(p) for p in y_vec])
        
        w_gen = self.E.gen()
        # Y_sym: y_vec'in E tabanına (1, w, w^2, ...) göre sembolik
        # toplamı; yani y vektörünün E cismindeki sembolik temsilidir.
        Y_sym = sum(y_vec_E[i] * (w_gen**i) for i in range(self.n))
        
        print(f"\n   --- [ARİTMETİK GÖRÜNTÜ] ---")
        print(f"   1. T(x) sonucu elde edilen y vektörü genişleme cismine (E) taşındı.")
        print(f"   2. E cismindeki sembolik polinom Y(x) oluşturuldu.")
        print(f"      Bu polinom 'w' (cisim üreteci) ve 'x_i' (girdi değişkenleri) cinsindendir:")
        print(f"   -> Y(x) = {Y_sym}")
        
        # C) Üs Alma (Merkez Dönüşüm)
        # Z = Y^e
        # MI merkez dönüşümünün sembolik olarak uygulanması: bu adım,
        # sistemin cebirsel "sertliğinin" (ve dolayısıyla klasik
        # lineerleştirme saldırısına konu olan bilineer ilişkilerin)
        # kaynağıdır.
        print(f"\n   3. Merkez Dönüşüm Uygulanıyor: Z = Y^{self.e}")
        Z_sym = Y_sym ** self.e
        
        # İndirgeme (Field Equations)
        # Genişleme cismi üzerinde hesaplanan Z_sym, F_q'nun cisim
        # özdeşliği (x_i^q = x_i) kullanılarak indirgenir; böylece
        # sonucun kuadratik dereceyi aşmaması sağlanır.
        vars_E = R_E_multi.gens()
        field_eqs_E = [vars_E[i]**self.q - vars_E[i] for i in range(self.n)]
        Ideal_E = R_E_multi.ideal(field_eqs_E)
        Z_reduced = Ideal_E.reduce(Z_sym)
        
        print(f"   -> Z(x) (İndirgenmiş Hali):")
        print(f"   {Z_reduced}")
        
        # D) Vektöre Dönüş (Map Down)
        # Z_reduced, E cismi katsayılı bir polinomdur. Bu katsayılar
        # tek tek F_q^n'e (extension_to_vector ile) geri açılarak,
        # MI(y)'nin F_q üzerindeki n bileşenli polinom vektör temsili
        # (mi_polys) elde edilir.
        mi_polys = [self.R(0) for _ in range(self.n)]
        for exps, coeff in Z_reduced.dict().items():
            coeff_vec = self.extension_to_vector(coeff) # E katsayısını F vektörüne çevir
            monomial = self.R.monomial(*exps)
            for k in range(self.n):
                mi_polys[k] += coeff_vec[k] * monomial
        MI_vec = vector(self.R, mi_polys)
        
        # E) S Dönüşümü
        # MI çekirdeğinin çıktısına gizli S afin dönüşümü uygulanarak
        # ham (tam, n bileşenli) açık anahtar hesaplanır ve cisim
        # idealine göre indirgenir.
        raw_pk = self.S_mat * MI_vec + self.S_vec
        Public_Key_Full = self.reduce_F_ideal(raw_pk)
        
        # 3. MINUS
        # SFLASH'ın karakteristik adımı: tam açık anahtarın son r
        # bileşeni atılır. Bu, saldırganın sistemin tam cebirsel
        # yapısına (n denklemin tamamına) erişimini engelleyerek,
        # MI'nin şifreleme bağlamındaki lineerleştirme saldırısına
        # karşı savunma sağlar.
        self.Public_Key_Minus = Public_Key_Full[:self.n - self.r]
        
        print(f"\n[1.3] Sonuç: Yayınlanan SFLASH Polinomları ({len(self.Public_Key_Minus)} adet):")
        for i, p in enumerate(self.Public_Key_Minus):
            print(f"   P_{i} = {p}")
            
        return self.Public_Key_Minus

    def sign(self, message_vec):
        """
        Verilen (n - r) boyutlu mesajı, SFLASH gizli anahtarını
        kullanarak imzalar.

        Algoritmanın Genel Akışı
        --------------------------
        SFLASH'ta imzalama, şifreleme şemalarının tersine bir yapıya
        sahiptir: imza üretimi, gizli anahtar sahibinin MI merkez
        dönüşümünün tersini (Y = W^d) alabilmesi üzerine kuruludur.
        "Minus" nedeniyle mesaj vektörü n boyutundan (n - r) boyutuna
        indirgenmiş olduğundan, imzalama öncesinde eksik r bileşen
        rastgele "padding" (dolgu) değerleriyle tamamlanmalıdır:

            1. y_full = mesaj || padding   (n boyutlu vektöre tamamlama)
            2. w = S^-1(y_full)             (gizli S dönüşümünün tersi)
            3. W = E cismindeki karşılığı (vector_to_extension)
            4. R = W^d                      (MI merkez dönüşümünün
                                              tersi; d, e'nin çarpımsal
                                              tersidir)
            5. r_vec = R'nin F_q^n vektör karşılığı
            6. imza = T^-1(r_vec)           (gizli T dönüşümünün tersi)

        Bu adımlar, tezin 5.1.5.1 bölümünde açıklanan "P = S o MI o T"
        bileşkesinin, gizli anahtar bilgisi kullanılarak adım adım
        tersine çevrilmesine karşılık gelir.

        Parametreler
        ------------
        message_vec : vector
            İmzalanacak mesaj; F_q üzerinde uzunluğu (n - r) olan
            vektör (örn. bir özet/hash fonksiyonunun çıktısı).

        Dönüş Değeri
        ------------
        vector
            Üretilen imza; F_q üzerinde uzunluğu n olan vektör.
        """
        print("\n" + "*"*40 + " ADIM 2: İMZALAMA (ARİTMETİK DETAYLI) " + "*"*40)
        print(f"   Mesaj: {message_vec}")
        
        # 1. Padding
        # "Minus" işlemiyle atılan r bileşenin yerini tutacak rastgele
        # dolgu değerleri üretilir; böylece S^-1 uygulanabilecek tam
        # boyutlu (n boyutlu) bir hedef vektör elde edilir.
        padding = random_vector(self.F, self.r)
        y_full = vector(self.F, list(message_vec) + list(padding))
        print(f"   Padding Eklendi (y): {y_full}")
        
        # 2. S^-1
        # Gizli S dönüşümünün tersi uygulanarak, MI çekirdeğinin
        # çıktısına karşılık gelen w vektörü elde edilir.
        w_vec = self.S_mat.inverse() * (y_full - self.S_vec)
        print(f"   S^-1 Sonucu (w): {w_vec}")
        
        # 3. MI Tersi (Asıl Aritmetik Kısmı)
        # Vektörü polinom (E elemanı) olarak gör
        W = self.vector_to_extension(w_vec)
        
        print(f"\n   --- [ARİTMETİK GÖRÜNTÜ] ---")
        print(f"   1. Vektör (w) Genişleme Cismine (E) taşındı:")
        print(f"      W(w) = {W}")
        print(f"      (Bu ifade GF({self.q}) katsayılı, 'w' değişkenli bir polinomdur)")
        
        print(f"   2. Ters Üs Alma İşlemi: R = W^{self.d}")
        # d, e'nin (q^n - 1) modülüne göre çarpımsal tersi olduğundan,
        # W^d işlemi MI merkez dönüşümünün (W = Y^e ise Y = W^d)
        # tersini verir. İmzalama işleminin özü bu adımdır.
        R = W ** self.d
        print(f"      R(w) = {R}")
        
        # Polinomu vektöre çevir
        r_vec = self.extension_to_vector(R)
        print(f"   3. Sonuç Tekrar Vektöre Çevrildi (r): {r_vec}")
        
        # 4. T^-1
        # Gizli T dönüşümünün tersi uygulanarak nihai imza elde edilir.
        signature = self.T_mat.inverse() * (r_vec - self.T_vec)
        print(f"\n   [SONUÇ] T^-1 ile İmza (x): {signature}")
        
        return signature

    def verify(self, message, signature):
        """
        Verilen imzanın, verilen mesaja karşılık gelip gelmediğini
        açık anahtar kullanarak doğrular.

        Teorik Gerekçe
        ---------------
        Doğrulama, imzalama sisteminin tanımı gereği, sadece açık
        (kamuya açık) bilgi kullanılarak gerçekleştirilebilir:
        imza (signature), yayınlanan açık anahtar polinomlarına
        (Public_Key_Minus) yerine konularak elde edilen çıktının,
        orijinal mesaja eşit olup olmadığı kontrol edilir. Bu, açık
        anahtarın P_pub(imza) = mesaj eşitliğini sağladığı, yani
        imzanın gizli anahtar sahibi tarafından üretilmiş olması
        gerektiği temeline dayanır.

        Parametreler
        ------------
        message : vector
            Doğrulanacak orijinal mesaj; F_q üzerinde uzunluğu
            (n - r) olan vektör.
        signature : vector
            Doğrulanacak imza; F_q üzerinde uzunluğu n olan vektör.

        Dönüş Değeri
        ------------
        bool
            İmza geçerliyse (P_pub(imza) == mesaj) True, aksi halde
            False.
        """
        print("\n" + "*"*40 + " ADIM 3: DOĞRULAMA " + "*"*40)
        # İmza vektörü, açık anahtar polinomlarındaki x_i
        # değişkenlerinin yerine konularak P_pub(imza) hesaplanır.
        val_dict = {self.vars[i]: signature[i] for i in range(self.n)}
        check_list = [p.subs(val_dict) for p in self.Public_Key_Minus]
        check_vec = vector(self.F, check_list)
        print(f"   P_pub(İmza) = {check_vec}")
        print(f"   Orijinal Mesaj = {message}")
        
        return check_vec == message




In [10]:
# ============================================================================
# SENARYO (GF(4), n=5, theta=1)
# ============================================================================
# Bu bölüm, yukarıda tanımlanan SFLASH sınıfının küçük,
# somut bir parametre seti (GF(4), n=5, r=2) üzerinde uçtan uca
# (anahtar üretimi -> imzalama -> doğrulama) test edilmesini sağlar.
# Amaç, tezin 5.1.5.1 bölümünde teorik olarak sunulan SFLASH (C*-Minus)
# yapısının doğru işlediğini doğrulamak ve ara adımları somut sayısal
# değerlerle izlenebilir kılmaktır.
print("\n>>> SFLASH SİMÜLASYONU <<<")

my_q = 4
my_n = 5
my_theta = 2
my_r = 2

sflash = SFLASH(q=my_q, n=my_n, theta=my_theta, r=my_r)
sflash.generate_keys()

# Test Mesajı: "Minus" nedeniyle mesaj boyutu (n - r) olmak zorundadır;
# çünkü açık anahtarın yalnızca bu kadar bileşeni yayınlanmaktadır.
msg = random_vector(sflash.F, my_n - my_r)
sig = sflash.sign(msg)

if sflash.verify(msg, sig):
    print("\n[BAŞARILI] İmza doğrulandı.")
else:
    print("\n[HATA] İmza geçersiz.")


>>> SFLASH SİMÜLASYONU <<<

SFLASH SİSTEMİ: 
Parametreler: GF(4), n=5, theta=2, r=2
Genişleme Cismi: F_4^5 (Üreteç 'w', Polinom: X^5 + a*X + 1)
[KURULUM] Gizli Üs e=17
[KURULUM] Ters Üs  d=662 (GCD=1)

**************************************** ADIM 1: ANAHTAR ÜRETİMİ  ****************************************

[1.1] Gizli Afin Dönüşümler:
   T Matrisi (5x5):
[    a     a     1     0     a]
[a + 1 a + 1 a + 1 a + 1     1]
[    a     0 a + 1     0 a + 1]
[a + 1 a + 1 a + 1     0 a + 1]
[a + 1 a + 1 a + 1     a a + 1]
   S Matrisi (5x5):
[a + 1     a     0     1 a + 1]
[    a a + 1 a + 1     0     0]
[    a     1     a a + 1     1]
[    a     0     0     a     1]
[a + 1     1     1 a + 1     0]
   c_T (Afin Sabit Vektörü): (0, 0, 0, a, a + 1)
   c_S (Afin Sabit Vektörü): (0, 1, a, 0, 0)

[1.2] Polinom Aritmetiği: P = S o MI o T

   --- [ARİTMETİK GÖRÜNTÜ] ---
   1. T(x) sonucu elde edilen y vektörü genişleme cismine (E) taşındı.
   2. E cismindeki sembolik polinom Y(x) oluşturuldu.
      B